# DR-Learner：从 GitHub 克隆并在 Vertex / 本地运行

本 notebook **不依赖** 本机 monorepo 路径：运行时从 GitHub 拉取代码，适合复制到 **Vertex AI Workbench** 或任意环境单独执行。

- **代码仓库**：[Jialu0901/lift-test-calibration](https://github.com/Jialu0901/lift-test-calibration)（`model_build` 根布局：`utils/`、`dr_learner_pipeline/`）
- **数据库**：在下方定义全局 **`DB_CONFIG`**（`host` / `user` / `password` / `database` / `port`），通过 **`run_pipeline_notebook(..., db_config=DB_CONFIG)`** 在进程内运行流水线，**无需** `db_config.json` 或 `LIFT_DB_CONFIG_JSON`。推荐用环境变量注入密码（见第 3 节）。
- **流水线参数**：第 **3b** 节在 notebook 内用 **`PIPELINE_CONFIG`**（完整 dict）+ 可选 **`PIPELINE_OVERRIDES`** 定义训练超参，**不读取** 仓库里的 `pipeline_grid.yaml`。命令行脚本仍可用 `--config` 指向 YAML。
- **私有仓库**：将 `REPO_URL` 改为带 PAT 的 HTTPS（或 SSH / `gh auth login`），勿在共享 notebook 中保留明文 token。


## 1. 配置克隆位置与分支

- `CLONE_ROOT`：克隆父目录。Vertex 上可设为 `Path("/home/jupyter")` 或环境变量 `LIFT_CLONE_ROOT`。
- 若目录已存在且为同一仓库，将执行 `git pull`。


In [21]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# 公开仓库默认 HTTPS；私有库请改用 SSH 或带凭证的 URL（勿提交含 token 的 notebook）
REPO_URL = os.environ.get(
    "LIFT_GITHUB_REPO_URL",
    "https://github.com/Jialu0901/lift-test-calibration.git",
)
REPO_DIR_NAME = os.environ.get("LIFT_REPO_DIR_NAME", "lift-test-calibration")
BRANCH = os.environ.get("LIFT_REPO_BRANCH", "main")

CLONE_ROOT = Path(os.environ.get("LIFT_CLONE_ROOT", str(Path.home() / "lift_repos"))).expanduser().resolve()
CLONE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = (CLONE_ROOT / REPO_DIR_NAME).resolve()

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
    print("Updated existing clone:", REPO_DIR)
else:
    if REPO_DIR.exists():
        raise FileExistsError(
            f"{REPO_DIR} exists and is not a git repo; remove it or change REPO_DIR_NAME / LIFT_CLONE_ROOT"
        )
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )
    print("Cloned to:", REPO_DIR)


From https://github.com/Jialu0901/lift-test-calibration
 * branch            main       -> FETCH_HEAD
Already on 'main'


Your branch is up to date with 'origin/main'.
Already up to date.
Updated existing clone: /Users/jialuliang/lift_repos/lift-test-calibration


From https://github.com/Jialu0901/lift-test-calibration
 * branch            main       -> FETCH_HEAD


In [22]:
import os
import sys
from pathlib import Path

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("cwd:", Path.cwd())
print("REPO_DIR:", REPO_DIR)


cwd: /Users/jialuliang/lift_repos/lift-test-calibration
REPO_DIR: /Users/jialuliang/lift_repos/lift-test-calibration


## 2. 安装依赖

在 **仓库根**（已 `chdir`）执行 `pip install`。


In [23]:
import subprocess
import sys

req = REPO_DIR / "requirements_dr_pipeline.txt"
if not req.is_file():
    raise FileNotFoundError(f"Missing {req} — check repo layout.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
    check=True,
    cwd=str(REPO_DIR),
)
print("pip install OK")


pip install OK


## 3. MySQL：`DB_CONFIG`（内存，无需 JSON 文件）

在下一格填写连接信息。**密码**请用环境变量（如 Vertex Secret 注入 `LIFT_MYSQL_PASSWORD`），勿把真实密钥写进要提交的 notebook。


In [ ]:
import os
from pathlib import Path

from utils.db_config import read_db_config_json

# 全局 dict，供 set_db_config_inline(DB_CONFIG) 使用。勿在要提交的 notebook 中写明文密码。
# 优先级：非空 LIFT_MYSQL_* 覆盖；密码来自 LIFT_MYSQL_PASSWORD → LIFT_MYSQL_PASSWORD_FILE 首行 → JSON。
# LIFT_DB_CONFIG_JSON 指向 JSON；未设置时尝试默认路径上的 db_config.json（与 CLI 一致）。
# 2003/(60)：host 不可达；1045：认证失败。
_base: dict = {}
try:
    if (os.environ.get("LIFT_DB_CONFIG_JSON") or "").strip():
        _base = read_db_config_json(os.environ["LIFT_DB_CONFIG_JSON"].strip())
    else:
        _base = read_db_config_json()
except FileNotFoundError:
    pass


def _pick_str(env_key: str, base_key: str, default: str) -> str:
    v = (os.environ.get(env_key) or "").strip()
    if v:
        return v
    b = str(_base.get(base_key) or "").strip()
    return b or default


_host = _pick_str("LIFT_MYSQL_HOST", "host", "internal-adb.workmagic.io")
_user = _pick_str("LIFT_MYSQL_USER", "user", "lift_instance")
_database = _pick_str("LIFT_MYSQL_DATABASE", "database", "workmagic_bi")

_port_s = (os.environ.get("LIFT_MYSQL_PORT") or "").strip()
if _port_s:
    _port = int(_port_s)
else:
    _pr = _base.get("port", 3306)
    _port = int(_pr) if _pr is not None else 3306

_pw = (os.environ.get("LIFT_MYSQL_PASSWORD") or "").strip()
_pw_file = (os.environ.get("LIFT_MYSQL_PASSWORD_FILE") or "").strip()
if not _pw and _pw_file:
    _pw = Path(_pw_file).expanduser().read_text(encoding="utf-8").splitlines()[0].strip()
if not _pw:
    _pw = str(_base.get("password") or "")

DB_CONFIG = {
    "host": _host,
    "user": _user,
    "password": _pw,
    "database": _database,
    "port": _port,
    "connection_timeout": 600,
    "consume_results": True,
}

if not DB_CONFIG["password"]:
    print(
        "Hint: set LIFT_MYSQL_PASSWORD, LIFT_MYSQL_PASSWORD_FILE, or LIFT_DB_CONFIG_JSON → db_config.json."
    )
else:
    print("DB_CONFIG ready (host=%r, user=%r, database=%r)" % (_host, _user, _database))


（可选）终端 **`python -m dr_learner_pipeline.run_pipeline`** 可传 **`--db-config-json`** 或 **`LIFT_DB_CONFIG_JSON`**。§3 的 **`DB_CONFIG`** 也会读取 **`LIFT_DB_CONFIG_JSON`** / 默认 **`db_config.json`** 并与 **`LIFT_MYSQL_*`** 合并。


## 3b. 参数总表（全内存：split + 完整流水线配置）

下一格定义 **`PIPELINE_CONFIG`**（与仓库内 `pipeline_grid.yaml` 等价结构，可自行修改）。**`PIPELINE_OVERRIDES`** 在其上做深度合并（如只改 `optuna_n_trials`）。**`SPLIT_DATES`**、**`DB_TABLE`** 亦在此。除 **GitHub 克隆**（代码 + `pip install -r requirements`）外不依赖其它本地文件。

训练前会把合并后的 `cfg` 写入临时 YAML，仅用于 `artifacts/` 记录；日常调参只需改本格 dict。


In [ ]:
import copy
import os
import tempfile
from pathlib import Path

# --- 日期划分（与 example_split_dates.json 同结构）---
SPLIT_DATES = {
    "train": [
        "2025-07-01",
        "2025-07-15",
        "2025-08-01",
        "2025-08-15",
        "2025-09-01",
        "2025-09-15",
        "2025-10-01",
        "2025-10-15",
        "2025-11-01",
        "2025-11-15",
        "2025-11-28",
    ],
    "val": ["2025-12-01", "2025-12-15"],
    "test": ["2025-12-25", "2026-01-01"],
}

# --- 完整流水线配置（原 pipeline_grid.yaml 内容，可按需增删键）---
PIPELINE_CONFIG: dict = {
    "random_seed": 42,
    "db_table": "workmagic_bi.lift_wide_scaled_v1",
    "wide_table_path": None,
    "sample_limit": None,
    "chunk_days": 1,
    "channels": None,
    "output_root": "output",
    "selected_features_csv": None,
    "feature_selection_protected_features": [
        "year",
        "month",
        "day",
        "quarter",
        "day_of_week",
        "is_weekday",
        "is_us_event",
        "event_pre_days",
        "event_post_days",
        "is_hist_order_missing",
        "is_hist_adview_missing",
    ],
    "feature_selection_protected_prefixes": [
        "day_name_",
        "us_event_name_",
        "us_event_type_",
        "emb_user_base_",
        "emb_event_",
    ],
    "n_cf_folds": 1,
    "propensity_candidates": ["lr"],
    "outcome_candidates": ["lgbm"],
    "optuna_n_trials": 15,
    "optuna_timeout": None,
    # "lead_families": ["lgbm", "xgb", "rf"],
    "lead_families": ["lgbm"],
    
    "n_rank_bins": 10,
    "placebo_shuffle_repeats": 20,
    "eval_placebo_on": "val",
    "refit_lead_on_train_val": False,
}

# 小范围覆盖（例如 {"optuna_n_trials": 30, "lead_families": ["lgbm"]}）
PIPELINE_OVERRIDES: dict = {}

# 若填写则覆盖 PIPELINE_CONFIG["db_table"]；None 表示用上一 dict 中的表名
DB_TABLE: str | None = None


## 4. 运行 DR-Learner 流水线（两步）

1. **加载 + 划分**：下一格用 **3b** 的 `PIPELINE_CONFIG`（不经 YAML 文件）构建 `cfg`，再 `load_wide_and_split(cfg, SPLIT_DATES)` 得到 **`bundle`**。
2. **训练**：再下一格 `run_pipeline_training_from_splits(..., SPLIT_DATES, ..., verbose=True)`，stdout 有 `[DR pipeline]` 进度。

**CLI**：终端仍用 `--config` + `--split_dates_path` JSON；与本 notebook 内存配置无关。

**若 `ImportError: load_wide_and_split` / `merge_pipeline_config`**：克隆目录过旧，对 **`REPO_DIR`** 执行 `git pull origin main`。


In [26]:
import copy
import inspect
import json
import os
import tempfile
from pathlib import Path

import yaml

from utils.db_config import set_db_config_inline

import dr_learner_pipeline.run_pipeline as _drp

_has_merge = hasattr(_drp, "merge_pipeline_config")
_load_params = list(inspect.signature(_drp.load_wide_and_split).parameters)
_train_params = list(
    inspect.signature(_drp.run_pipeline_training_from_splits).parameters
)
_load_uses_split_spec = "split_spec" in _load_params
_train_uses_split_spec = "split_spec" in _train_params

if not hasattr(_drp, "load_wide_and_split"):
    raise RuntimeError(
        f"克隆目录里的 run_pipeline.py 缺少 load_wide_and_split。\n"
        f"请执行: cd {REPO_DIR} && git pull origin main\n"
        "（若新代码只在本地 monorepo，请先 push 再拉取。）"
    )


def _merge_fallback(base: dict, overrides: dict) -> dict:
    out = copy.deepcopy(base)
    for k, v in (overrides or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _merge_fallback(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


if _has_merge:
    from dr_learner_pipeline.run_pipeline import merge_pipeline_config
else:
    merge_pipeline_config = _merge_fallback
    print(
        "[notebook] clone 无 merge_pipeline_config，已用内置递归合并代替；建议 git pull。"
    )

from dr_learner_pipeline.run_pipeline import load_wide_and_split

set_db_config_inline(DB_CONFIG)

cfg = merge_pipeline_config(copy.deepcopy(PIPELINE_CONFIG), PIPELINE_OVERRIDES)
if DB_TABLE:
    cfg["db_table"] = DB_TABLE.strip()

_split_is_dict = isinstance(SPLIT_DATES, dict)
_split_json_path: Path | None = None


def _ensure_split_json() -> Path:
    global _split_json_path
    if _split_json_path is None:
        _fd_sp, _split_p = tempfile.mkstemp(suffix=".json", prefix="dr_nb_split_")
        os.close(_fd_sp)
        _split_json_path = Path(_split_p)
        _split_json_path.write_text(
            json.dumps(SPLIT_DATES, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
    return _split_json_path


if _split_is_dict:
    _load_arg = SPLIT_DATES if _load_uses_split_spec else _ensure_split_json()
    RUN_SPLIT_FOR_TRAINING = (
        SPLIT_DATES if _train_uses_split_spec else _ensure_split_json()
    )
else:
    _p = Path(SPLIT_DATES) if not isinstance(SPLIT_DATES, Path) else SPLIT_DATES
    _load_arg = _p
    RUN_SPLIT_FOR_TRAINING = _p

if _split_is_dict and not _load_uses_split_spec:
    print(
        "[notebook] 克隆的 load_wide_and_split 仅接受 Path：已将 SPLIT_DATES 写入临时 JSON。"
    )

# 供 artifacts 复制的配置快照（临时文件，非仓库内 pipeline_grid.yaml）
_fd, _snap = tempfile.mkstemp(suffix=".yaml", prefix="dr_nb_cfg_")
os.close(_fd)
CONFIG_SNAPSHOT_PATH = Path(_snap)
with open(CONFIG_SNAPSHOT_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)

bundle = load_wide_and_split(cfg, _load_arg)
print(
    "df shape:",
    bundle.df.shape,
    "| load window:",
    bundle.date_load_start,
    "..",
    bundle.date_load_end,
)
print("train", len(bundle.train_df), "val", len(bundle.val_df), "test", len(bundle.test_df))
print("cfg snapshot:", CONFIG_SNAPSHOT_PATH)


df shape: (3723573, 218) | load window: 2025-07-01 .. 2026-01-01
train 2823939 val 464868 test 434766
cfg snapshot: /var/folders/nh/lk_mqzl50zn_7c5jv8rt6l680000gn/T/dr_nb_cfg_8vbbrg3b.yaml


### 4b. 训练（使用上一格的 `cfg` 与 `bundle`）

In [27]:
import inspect as _inspect

from dr_learner_pipeline.run_pipeline import run_pipeline_training_from_splits

cli_overrides: dict = {}
if DB_TABLE:
    cli_overrides["db_table"] = DB_TABLE.strip()

_train_params = list(_inspect.signature(run_pipeline_training_from_splits).parameters)
if "split_spec" not in _train_params and isinstance(RUN_SPLIT_FOR_TRAINING, Path):
    cli_overrides.setdefault("split_dates_path", str(RUN_SPLIT_FOR_TRAINING.resolve()))

exp_dir = run_pipeline_training_from_splits(
    cfg,
    RUN_SPLIT_FOR_TRAINING,
    CONFIG_SNAPSHOT_PATH,
    cli_overrides,
    bundle,
    verbose=True,
)
print("Done:", exp_dir)


2026-03-26 03:45:48,846 INFO [dr_learner_pipeline.run_pipeline] Experiment directory /Users/jialuliang/lift_repos/lift-test-calibration/output/20260326_034548


[DR pipeline] experiment directory: /Users/jialuliang/lift_repos/lift-test-calibration/output/20260326_034548


2026-03-26 03:45:48,851 INFO [dr_learner_pipeline.run_pipeline] Training phase: rows train=2823939 val=464868 test=434766 (load range 2025-07-01 .. 2026-01-01)


[DR pipeline] training phase: train=2823939 val=464868 test=434766 (loaded 2025-07-01 .. 2026-01-01)


2026-03-26 03:45:48,854 INFO [dr_learner_pipeline.run_pipeline] Feature list: n=205 protected_in_model=121 source=all_candidates


...

2026-03-26 05:08:03,651 INFO [dr_learner_pipeline.stages.lead_optuna] Best lead estimator: lgbm val_mse=3.563437
2026-03-26 05:08:03,657 INFO [dr_learner_pipeline.run_pipeline] Channel microsoft lead: lgbm params={'family': 'lgbm', 'n_estimators': 253, 'max_depth': 7, 'learning_rate': 0.052024457297037606, 'num_leaves': 17, 'val_mse': 3.5634369109720248}


[DR pipeline] channel microsoft: lead OK (family=lgbm)
[DR pipeline] channel microsoft: test predictions + metrics + placebo...
[DR pipeline] channel microsoft: done
[DR pipeline] channel snapchat: start


2026-03-26 05:08:06,140 INFO [dr_learner_pipeline.run_pipeline] Channel snapchat feature list: {'source': 'all_candidates', 'n_protected_in_model': 121, 'n_selected': 205}


[DR pipeline] channel snapchat: n_features=205; nuisance/base selection...


KeyboardInterrupt: 

## 产物与解析

- 输出目录：`REPO_DIR / "output" / <时间戳> /`
- 结果解析：若 Git 仓库中包含 **`parse_experiment_results.ipynb`**（可能在 `dr_learner_pipeline/notebook/` 或 `notebooks/`），克隆后打开并设置 `EXP_DIR` 为本次 `output/<时间戳>`。
